In [1]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
import wfdb
import numpy as np
from scipy.signal import butter, filtfilt

# 1. 44 records (102, 104, 107, 217 excluded as requested)
all_records = [
    '100', '101', '103', '105', '106', '108', '109', '111', '112', '113',
    '114', '115', '116', '117', '118', '119', '121', '122', '123', '124',
    '200', '201', '202', '203', '205', '207', '208', '209', '210', '212',
    '213', '214', '215', '219', '220', '221', '222', '223', '228', '230',
    '231', '232', '233', '234'
]

os.makedirs('ecg_data', exist_ok=True)
print("Downloading 44 records from MIT-BIH...")
for rec in all_records:
    if not os.path.exists(f'ecg_data/{rec}.dat'):
        wfdb.dl_database('mitdb', dl_dir='ecg_data', records=[rec])
print("All 44 records downloaded successfully!")

# 2. Bandpass filter (0.5 to 45 Hz)
def butter_bandpass_filter(data, lowcut=0.5, highcut=45.0, fs=360.0, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

# 3. Label Mapping
normal_symbols = {'N', 'L', 'R', 'e', 'j'}
abnormal_symbols = {'A', 'a', 'J', 'S', 'V', 'E', 'F', '/', 'f', 'Q'}

# 4. Patient-wise / Record-wise Split (To prevent data leakage)
train_records = [
    '100', '101', '103', '105', '106', '108', '109', '111', '112', '113',
    '114', '115', '116', '117', '118', '119', '121', '122', '123', '124',
    '200', '201', '202', '205', '208', '209', '210', '212', '213', '215'
]
val_records  = ['203', '214', '219', '220', '221', '222', '228']
test_records = ['207', '223', '230', '231', '232', '233', '234']

window_size = 180  # 180 left + 180 right = 360 samples (exact 1 second window)

def process_records(record_list):
    X_list, y_list = [], []
    for rec in record_list:
        record = wfdb.rdrecord(f'ecg_data/{rec}')
        raw_signal = record.p_signal[:, 0]  # Channel 0 (MLII lead)
        filtered_signal = butter_bandpass_filter(raw_signal)
        annotation = wfdb.rdann(f'ecg_data/{rec}', 'atr')

        for idx, sym in zip(annotation.sample, annotation.symbol):
            if sym in normal_symbols:
                lbl = 0
            elif sym in abnormal_symbols:
                lbl = 1
            else:
                continue  # Non-beat symbols ignore

            if idx - window_size >= 0 and idx + window_size < len(filtered_signal):
                segment = filtered_signal[idx - window_size : idx + window_size]
                std_val = np.std(segment)
                if std_val > 0:
                    segment = (segment - np.mean(segment)) / std_val
                else:
                    segment = segment - np.mean(segment)
                X_list.append(segment)
                y_list.append(lbl)

    X_arr = np.array(X_list, dtype=np.float32)
    X_arr = np.expand_dims(X_arr, axis=-1)  # Shape (N, 360, 1)
    y_arr = np.array(y_list, dtype=np.int32)
    return X_arr, y_arr

print("\nProcessing beats for Train, Val and Test...")
X_train, y_train = process_records(train_records)
X_val, y_val     = process_records(val_records)
X_test, y_test   = process_records(test_records)

# 5. Save as compressed .npz exactly as teammate requested
np.savez_compressed(
    'ecg_processed_v1.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)

print("\n--- ecg_processed_v1.npz Successfully Saved! ---")
print(f"Train set: {X_train.shape}, Normal: {np.sum(y_train == 0)}, Abnormal: {np.sum(y_train == 1)}")
print(f"Val set:   {X_val.shape}, Normal: {np.sum(y_val == 0)}, Abnormal: {np.sum(y_val == 1)}")
print(f"Test set:  {X_test.shape}, Normal: {np.sum(y_test == 0)}, Abnormal: {np.sum(y_test == 1)}")

Generating record list for: 100
Generating list of all files for: 100
Finished downloading files
Generating record list for: 101
Generating list of all files for: 101
Finished downloading files
Generating record list for: 103
Generating list of all files for: 103
Finished downloading files
Generating record list for: 105
Generating list of all files for: 105
Finished downloading files
Generating record list for: 106
Generating list of all files for: 106
Finished downloading files
Generating record list for: 108
Generating list of all files for: 108
Finished downloading files
Generating record list for: 109
Generating list of all files for: 109
Finished downloading files
Generating record list for: 111
Generating list of all files for: 111
Finished downloading files
Generating record list for: 112
Generating list of all files for: 112
Finished downloading files
Generating record list for: 113
Generating list of all files for: 113
Finished downloading files
Generating record list for: 11